<a href="https://colab.research.google.com/github/shentan-shiina/Colab_Deployment_Log/blob/main/Create_LIBERO_Env.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀Create Your Custom Conda Environment in Colab

## Connect to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Install [konda](https://github.com/tamnguyenvan/konda) for env management

In [ ]:
!pip install konda
import konda
konda.install()
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

Installing Miniconda to /usr/local...
✅ Miniconda installed successfully!
Run '!conda --version' to check if conda is working.

📋 Usage examples:
  konda create -n my_env python=3.11 -y
  konda activate my_env
accepted Terms of Service for https://repo.anaconda.com/pkgs/main
accepted Terms of Service for https://repo.anaconda.com/pkgs/r


## Create conda env on session machine (*Temporary, upload to GoogleDrive when finished*)

In [ ]:
%%bash
konda create -n libero python=3.8.13 -y
source activate libero
git clone https://github.com/Lifelong-Robot-Learning/LIBERO.git
cd LIBERO
pip install -r requirements.txt
pip install torch==1.11.0+cu113 torchvision==0.12.0+cu113 torchaudio==0.11.0 --extra-index-url https://download.pytorch.org/whl/cu113
pip install -e .

Jupyter detected...
2 channel Terms of Service accepted
Retrieving notices: - \ | / - \ | / - \ | / - \ | / - \ | / - \ | / done
Channels:
 - defaults
Platform: linux-64
Solving environment: | done

## Package Plan ##

  environment location: /usr/local/envs/libero

  added / updated specs:
    - python=3.8.13


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    libffi-3.3                 |       he6710b0_2          50 KB
    openssl-1.1.1w             |       h7f8727e_0         3.7 MB
    pip-24.2                   |   py38h06a4308_0         2.2 MB
    python-3.8.13              |       haa1d7c7_1        20.2 MB
    setuptools-75.1.0          |   py38h06a4308_0         1.7 MB
    sqlite-3.51.1              |       he0a8d7e_0         1.2 MB
    wheel-0.44.0               |   py38h06a4308_0         108 KB
    ---------------------------------



==> WARNING: A newer version of conda exists. <==
    current version: 25.11.1
    latest version: 26.1.0

Please update conda by running

    $ conda update -n base -c defaults conda


Cloning into 'LIBERO'...
Updating files: 100% (1116/1116), done.
  DEPRECATION: Legacy editable install of libero==0.1.0 from file:///content/LIBERO (setup.py develop) is deprecated. pip 25.0 will enforce this behaviour change. A possible replacement is to add a pyproject.toml or enable --use-pep517, and use setuptools >= 64. If the resulting installation is not behaving as expected, try using --config-settings editable_mode=compat. Please consult the setuptools documentation for more information. Discussion can be found at https://github.com/pypa/pip/issues/11457


## Use the custom env (*Remember to source activate the env in each cell*)

In [ ]:
%cd LIBERO
!ls

/content/LIBERO
benchmark_scripts  libero.egg-info  README.md	      setup.py
images		   LICENSE	    requirements.txt  templates
libero		   notebooks	    scripts


In [ ]:
!source activate libero && python benchmark_scripts/download_libero_datasets.py

Do you want to specify a custom path for the dataset folder? (Y/N): Traceback (most recent call last):

^C


## You may need this for path correction

In [ ]:
import sys
sys.path.append("/usr/local/envs/libero/lib/python3.8/site-packages")
sys.path.append("/usr/local/envs/libero")

## Write .py files and save them, and you can execute under your env!

In [ ]:
%%writefile check_dataset.py
import os
from libero.libero import benchmark
from libero.libero.envs import OffScreenRenderEnv
from libero.libero import benchmark, get_libero_path, set_libero_default_path
import matplotlib.pyplot as plt

benchmark_dict = benchmark.get_benchmark_dict()
task_suite_name = "libero_goal" # can also choose libero_spatial, libero_object, etc.
task_suite = benchmark_dict[task_suite_name]()

# retrieve a specific task
task_id = 0
task = task_suite.get_task(task_id)
task_name = task.name
task_description = task.language
task_bddl_file = os.path.join(get_libero_path("bddl_files"), task.problem_folder, task.bddl_file)
print(f"[info] retrieving task {task_id} from suite {task_suite_name}, the " + \
      f"language instruction is {task_description}, and the bddl file is {task_bddl_file}")

# step over the environment
env_args = {
    "bddl_file_name": task_bddl_file,
    "camera_heights": 128,
    "camera_widths": 128
}
env = OffScreenRenderEnv(**env_args)
env.seed(0)
env.reset()
init_states = task_suite.get_task_init_states(task_id) # for benchmarking purpose, we fix the a set of initial states
init_state_id = 0
env.set_init_state(init_states[init_state_id])

dummy_action = [0.] * 7
for step in range(10):
    obs, reward, done, info = env.step(dummy_action)
    print(obs)

env.close()

Writing check_dataset.py


In [ ]:
!source activate libero && python check_dataset.py

[robosuite WARNING] No private macro file found! (__init__.py:7)
[robosuite WARNING] It is recommended to use a private macro file (__init__.py:8)
[robosuite WARNING] To setup, run: python /usr/local/envs/libero/lib/python3.8/site-packages/robosuite/scripts/setup_macros.py (__init__.py:9)
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
[info] retrieving task 0 from suite libero_goal, the language instruction is open the middle drawer of the cabinet, and the bddl file is /content/LIBERO/libero/libero/./bddl_files/libero_goal/open_the_middle_drawer_of_the_cabinet.bddl
OrderedDict([('robot0_joint_pos', array([ 0.00980392, -0

## Copy/Upload the env, workspace to your GoogleDrive, then you are good to deploy! 🤙

In [ ]:
!ls /usr/local/envs/libero

bin		 conda-meta  include  share  x86_64-conda-linux-gnu
compiler_compat  etc	     lib      ssl


In [ ]:
!tar -czvf /content/drive/MyDrive/Colab/1Environments/libero.tar.gz -C /usr/local/envs libero
!cp /content/drive/MyDrive/Colab/1Environments/libero.tar.gz /content/

Streaming output truncated to the last 5000 lines.
libero/lib/python3.8/site-packages/torch/include/ATen/ops/_standard_gamma_grad_native.h
libero/lib/python3.8/site-packages/torch/include/ATen/ops/_embedding_bag_backward_native.h
libero/lib/python3.8/site-packages/torch/include/ATen/ops/logsumexp_native.h
libero/lib/python3.8/site-packages/torch/include/ATen/ops/leaky_relu_backward_cpu_dispatch.h
libero/lib/python3.8/site-packages/torch/include/ATen/ops/_cufft_clear_plan_cache.h
libero/lib/python3.8/site-packages/torch/include/ATen/ops/new_empty_compositeexplicitautograd_dispatch.h
libero/lib/python3.8/site-packages/torch/include/ATen/ops/avg_pool3d_backward_compositeexplicitautograd_dispatch.h
libero/lib/python3.8/site-packages/torch/include/ATen/ops/fractional_max_pool2d_ops.h
libero/lib/python3.8/site-packages/torch/include/ATen/ops/mv_ops.h
libero/lib/python3.8/site-packages/torch/include/ATen/ops/_new_zeros_with_same_feature_meta.h
libero/lib/python3.8/site-packages/torch/include/

## 💅🏻 Create a magic command for easier development

In [ ]:
!pip install IPython

  Using cached traitlets-5.14.3-py3-none-any.whl.metadata (10 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 622.8/622.8 kB 3.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 7.0 MB/s  0:00:00
Using cached traitlets-5.14.3-py3-none-any.whl (85 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15/15 [IPython]


In [ ]:
import sys
import tempfile
import os
from IPython.core.magic import register_cell_magic

# --- CONFIGURATION ---
# Set your default environment name here
DEFAULT_ENV = "libero"
# ---------------------

@register_cell_magic
def py_env(line, cell):
    """
    Executes the cell content inside a specific Conda environment.
    Usage:
      %%run_env
      (uses DEFAULT_ENV)

      or

      %%run_env other_env_name
    """
    env_name = line.strip() if line.strip() else DEFAULT_ENV

    # 1. Create a temporary python file
    # We use delete=False because we need to close the file before executing it
    with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False) as f:
        f.write(cell)
        script_path = f.name

    try:
        # 2. Execute the script inside the Conda environment
        print(f"Executing block in '{env_name}'...")
        !source activate {env_name} && python {script_path}
    finally:
        # 3. Clean up the temporary file
        os.remove(script_path)


In [ ]:
import sys
import tempfile
import os
from IPython.core.magic import register_line_cell_magic
from IPython import get_ipython

# --- CONFIGURATION ---
# Set your default environment name here
DEFAULT_ENV = "libero"
# ---------------------

@register_line_cell_magic
def bash_env(line, cell=None):
    """
    Executes shell commands inside a specific Conda environment.

    Line magic usage (like '!'):
      %cmd_in_env pip list
      %cmd_in_env myenv pip list

    Cell magic usage (like '%%bash'):
      %%cmd_in_env
      echo "Installing..."
      pip install pandas
    """
    # Parse arguments
    parts = line.split()

    # Check if the first argument is a known environment, otherwise use default
    # Note: This is a simple heuristic. If you name your env "pip", this might get confused.
    # A safer way is to assume DEFAULT_ENV unless specified, but for flexibility:
    if parts and not parts[0].startswith("-"):
        # You might want to add logic here to check if parts[0] is actually an env
        # For this snippet, we will assume if the user provides 2+ args, first is env
        # If 1 arg, use default.
        pass

    # For simplicity, let's strictly use DEFAULT_ENV to avoid command confusion
    # unless you explicitly modify the logic below.
    env_name = DEFAULT_ENV
    command_line = line

    ipy = get_ipython()

    if cell is None:
        # --- LINE MAGIC (Replaces '!') ---
        # Usage: %cmd_in_env pip list
        print(f"Executing in '{env_name}': {command_line}")
        ipy.system(f"source activate {env_name} && {command_line}")

    else:
        # --- CELL MAGIC (Replaces '%%bash') ---
        # Usage: %%cmd_in_env
        print(f"Executing block in '{env_name}'...")

        # Write the cell content to a temporary bash script
        with tempfile.NamedTemporaryFile(mode='w', suffix='.sh', delete=False) as f:
            f.write("set -e\n") # Stop on error
            f.write(cell)
            script_path = f.name

        try:
            # Run the script using bash inside the conda env
            ipy.system(f"source activate {env_name} && /bin/bash {script_path}")
        finally:
            os.remove(script_path)

## Test your env dependencies

In [ ]:
import torch
print(torch.__version__)

2.9.0+cpu


In [ ]:
%debug
!source activate libero
import torch
print(torch.__version__)

ERROR:root:No traceback has been produced, nothing to debug.


2.9.0+cpu


In [ ]:
%%py_env
import torch
print(torch.__version__)

Executing block in 'libero'...
1.11.0+cu113
